<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/tools/bilig_workpaper_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bilig WorkPaper MCP with LlamaIndex

This notebook uses LlamaIndex's MCP client to run a local [Bilig WorkPaper](https://github.com/proompteng/bilig) server over stdio. The server exposes a spreadsheet-style formula workbook as tools: read cells, write inputs, recalculate formulas, persist JSON, and export the resulting WorkPaper document.

The example does not require an LLM key. It is a direct tool smoke test that an agent workflow can build on.

In [ ]:
%pip install llama-index-tools-mcp

## Start the WorkPaper MCP server

`BasicMCPClient` can launch stdio MCP servers directly. Here `npm exec` installs the published Bilig package, starts `bilig-workpaper-mcp`, and creates a writable demo quote workbook in a temporary directory.

In [ ]:
import json
import tempfile
from pathlib import Path

from llama_index.tools.mcp import BasicMCPClient, aget_tools_from_mcp_url

work_dir = Path(tempfile.mkdtemp(prefix="bilig-llamaindex-"))
workpaper_path = work_dir / "quote.workpaper.json"

client = BasicMCPClient(
    "npm",
    args=[
        "exec",
        "--yes",
        "--package",
        "@bilig/workpaper@0.51.1",
        "--",
        "bilig-workpaper-mcp",
        "--workpaper",
        str(workpaper_path),
        "--init-demo-workpaper",
        "--writable",
    ],
    timeout=60,
)

print(workpaper_path)

## Discover and call tools

The workbook starts with an `Inputs` sheet and a `Summary` sheet. `Summary!B3` is a formula cell for expected ARR.

In [ ]:
tools = await client.list_tools()
print([tool.name for tool in tools.tools])

In [ ]:
async def call_json(tool_name: str, arguments: dict | None = None) -> dict:
    result = await client.call_tool(tool_name, arguments or {})
    text_items = [item.text for item in result.content if item.type == "text"]
    if not text_items:
        raise ValueError(f"{tool_name} returned no text content")
    return json.loads(text_items[0])


inputs = await call_json("read_range", {"range": "Inputs!A1:B5"})
summary = await call_json("read_range", {"range": "Summary!A1:B5"})

print(inputs["serialized"])
print(summary["serialized"])

## Edit an input and verify formula readback

Change the win rate from `0.25` to `0.4`. The expected customer count changes from `5` to `8`, and expected ARR changes from `60000` to `96000`.

In [ ]:
before_arr = await call_json(
    "read_cell", {"sheetName": "Summary", "address": "B3"}
)

edit = await call_json(
    "set_cell_contents",
    {"sheetName": "Inputs", "address": "B3", "value": 0.4},
)

after_customers = await call_json(
    "read_cell", {"sheetName": "Summary", "address": "B2"}
)
after_arr = await call_json(
    "read_cell", {"sheetName": "Summary", "address": "B3"}
)

print("ARR before:", before_arr["value"]["value"])
print("Customers after:", after_customers["value"]["value"])
print("ARR after:", after_arr["value"]["value"])
print("Persisted:", edit["checks"]["persisted"])
print("Restored matches after:", edit["checks"]["restoredMatchesAfter"])

assert before_arr["value"]["value"] == 60000
assert after_customers["value"]["value"] == 8
assert after_arr["value"]["value"] == 96000
assert edit["checks"]["persisted"] is True
assert edit["checks"]["restoredMatchesAfter"] is True

## Export the WorkPaper document

The exported JSON can be saved, reviewed, or passed to another agent. The same workbook file is also persisted on disk by the write tool.

In [ ]:
exported = await call_json("export_workpaper_document", {"includeConfig": True})

print("Exported bytes:", exported["serializedBytes"])
print("File exists:", workpaper_path.exists())
print("File bytes:", workpaper_path.stat().st_size)

assert exported["document"]["sheets"][0]["content"][2][1] == 0.4
assert workpaper_path.exists()

## Convert the MCP server to LlamaIndex tools

For an agent, load the same MCP server as LlamaIndex `FunctionTool`s and pass them to your workflow or agent of choice.

In [ ]:
llamaindex_tools = await aget_tools_from_mcp_url(
    "npm",
    client=client,
    allowed_tools=["read_cell", "set_cell_contents", "export_workpaper_document"],
)

print([tool.metadata.name for tool in llamaindex_tools])